# Force Validation Analysis
Combines loadcell, gui, camera streams → 100 Hz (nearest-neighbour) → steady-state extraction → combined CSV

In [1]:
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── session to analyse ────────────────────────────────────────────────────────
SESSION_DIR  = "/home/sujith/Documents/NOARK_backbone/csv_data/trial_1_20260612_154538"
OUTPUT_NAME  = "pos_1_trial_1"          # output file name (no extension)
OUTPUT_DIR   = "/home/sujith/Documents/NOARK_backbone/csv_data/trial_1_error_analysis"  # output directory (must exist)

# ── analysis parameters ───────────────────────────────────────────────────────
RESAMPLE_HZ     = 100          # target rate
RESAMPLE_PERIOD = f"{1000//RESAMPLE_HZ}ms"
NN_TOLERANCE    = pd.Timedelta("5ms")   # nearest-neighbour search window

HOLD_TIME_S     = 2.0          # sweep step hold duration (must match firmware)
SETTLE_TOL      = 0.10         # 10 % band around target magnitude
SETTLE_MIN_S    = 1.0          # must stay in band for ≥ this many seconds
# 0N steps no longer generated (removed from MAG_LIST); flag kept for old sessions
SKIP_ZERO_N     = True         # skip steps where target magnitude == 0

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1 — Load raw CSVs

In [2]:
def load_ts(path, **kwargs):
    df = pd.read_csv(path, **kwargs)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp").sort_index()
    return df

df_lc      = load_ts(os.path.join(SESSION_DIR, "loadcell.csv"))
df_gui     = load_ts(os.path.join(SESSION_DIR, "gui.csv"))
df_cam     = load_ts(os.path.join(SESSION_DIR, "camera.csv"))
df_sweep   = load_ts(os.path.join(SESSION_DIR, "sweep_targets.csv"))

print(f"loadcell   : {len(df_lc):6d} rows  ({len(df_lc)/(df_lc.index[-1]-df_lc.index[0]).total_seconds():.1f} Hz)")
print(f"gui        : {len(df_gui):6d} rows  ({len(df_gui)/(df_gui.index[-1]-df_gui.index[0]).total_seconds():.1f} Hz)")
print(f"sweep steps: {len(df_sweep):6d}")
print(f"camera     : {len(df_cam):6d} rows")

loadcell   :   9823 rows  (137.0 Hz)
gui        :   9894 rows  (138.0 Hz)
sweep steps:     28
camera     :      1 rows


## 2 — Resample to 100 Hz (nearest-neighbour, no interpolation)

In [3]:
# Common 100 Hz grid covering the full session
t_start = min(df_lc.index[0], df_gui.index[0])
t_end   = max(df_lc.index[-1], df_gui.index[-1])
idx_100 = pd.date_range(t_start, t_end, freq=RESAMPLE_PERIOD)

def nearest_resample(df, idx, tol):
    """For each tick in idx pick the actual nearest sample within tol."""
    return df.reindex(idx, method="nearest", tolerance=tol)

lc_100  = nearest_resample(df_lc,  idx_100, NN_TOLERANCE)
gui_100 = nearest_resample(df_gui, idx_100, NN_TOLERANCE)

# Camera is a single reference position — broadcast to all rows
pos_x = df_cam["pos_x"].iloc[0]
pos_z = df_cam["pos_z"].iloc[0]

print(f"100 Hz grid: {len(idx_100)} ticks  ({(t_end-t_start).total_seconds():.1f} s)")
print(f"lc  valid  : {lc_100.notna().all(axis=1).sum()} / {len(idx_100)}")
print(f"gui valid  : {gui_100.notna().all(axis=1).sum()} / {len(idx_100)}")

100 Hz grid: 7171 ticks  (71.7 s)
lc  valid  : 5032 / 7171
gui valid  : 6886 / 7171


## 3 — Compute quantities at 100 Hz

In [7]:
# ── Measured (load cell) ──────────────────────────────────────────────────────
Fxl = lc_100["Fx"]
Fyl = lc_100["Fy"]
Fml = np.hypot(Fxl, Fyl)
Fdl = np.degrees(np.arctan2(Fyl, Fxl))

# ── Commanded (gui) ───────────────────────────────────────────────────────────
# Support both old format (magnitude+direction only) and new format (with Fx/Fz)
if "Fx" in gui_100.columns and "Fz" in gui_100.columns:
    Fxg = gui_100["Fx"]
    Fzg = gui_100["Fz"]
    Fmg = gui_100["magnitude"]
    Fdg = gui_100["direction"]
else:
    Fmg = gui_100["magnitude"]
    Fdg = gui_100["direction"]
    Fxg = Fmg * np.cos(np.radians(Fdg))
    Fzg = Fmg * np.sin(np.radians(Fdg))

# ── Errors ────────────────────────────────────────────────────────────────────
mag_error = np.abs(Fml - Fmg)
dir_error = np.abs(Fdl - Fdg)
dir_error = dir_error.where(dir_error <= 180, 360 - dir_error)   # wrap to [0, 180]

me_percent = np.where(Fmg > 1e-6, (mag_error / Fmg) * 100, np.nan)
de_percent = (dir_error / 360.0) * 100
print(mag_error)
print(dir_error)
print(me_percent)
print(de_percent)
print("Derived quantities computed.")

2026-06-12 15:47:53.678787     0.995125
2026-06-12 15:47:53.688787          NaN
2026-06-12 15:47:53.698787     3.450022
2026-06-12 15:47:53.708787          NaN
2026-06-12 15:47:53.718787     4.164957
                                ...    
2026-06-12 15:49:05.338787    22.373951
2026-06-12 15:49:05.348787          NaN
2026-06-12 15:49:05.358787          NaN
2026-06-12 15:49:05.368787          NaN
2026-06-12 15:49:05.378787    22.605769
Freq: 10ms, Length: 7171, dtype: float64
2026-06-12 15:47:53.678787    176.254864
2026-06-12 15:47:53.688787           NaN
2026-06-12 15:47:53.698787    133.671498
2026-06-12 15:47:53.708787           NaN
2026-06-12 15:47:53.718787     34.945644
                                 ...    
2026-06-12 15:49:05.338787    111.981400
2026-06-12 15:49:05.348787           NaN
2026-06-12 15:49:05.358787           NaN
2026-06-12 15:49:05.368787           NaN
2026-06-12 15:49:05.378787    110.122761
Freq: 10ms, Length: 7171, dtype: float64
[        nan         nan 69

## 4 — Steady-state detection per sweep step

In [8]:
# Only non-zero target steps
steps = df_sweep[df_sweep["target_mag_N"] > 0].copy() if SKIP_ZERO_N else df_sweep.copy()
steps = steps[steps["feasible"] == 1]

SETTLE_N = int(SETTLE_MIN_S * RESAMPLE_HZ)   # minimum samples in band

def find_settle_time(step_start, step_end, target_mag, Fml_series):
    """
    Within [step_start, step_end], find the first time Fml enters
    a ±SETTLE_TOL band around target_mag and stays for SETTLE_MIN_S.
    Returns (t_settle, valid) — t_settle is a pd.Timestamp.
    """
    window = Fml_series.loc[step_start:step_end].dropna()
    if len(window) == 0:
        return None, False
    lo = target_mag * (1 - SETTLE_TOL)
    hi = target_mag * (1 + SETTLE_TOL)
    in_band = (window >= lo) & (window <= hi)
    # Find first run of ≥ SETTLE_N consecutive True values
    streak = 0
    for ts, val in in_band.items():
        if val:
            streak += 1
            if streak >= SETTLE_N:
                # t_settle = this timestamp minus (SETTLE_N-1) ticks
                idx_pos = in_band.index.get_loc(ts)
                t_settle = in_band.index[idx_pos - SETTLE_N + 1]
                return t_settle, True
        else:
            streak = 0
    return None, False

step_records = []
for i, (ts, row) in enumerate(steps.iterrows()):
    step_start = ts
    # Step end = next step start or step_start + HOLD_TIME
    if i + 1 < len(steps):
        step_end = steps.index[i + 1]
    else:
        step_end = step_start + pd.Timedelta(seconds=HOLD_TIME_S)

    t_settle, valid = find_settle_time(step_start, step_end, row["target_mag_N"], Fml)
    step_records.append({
        "step"         : int(row["step"]),
        "step_start"   : step_start,
        "step_end"     : step_end,
        "target_mag"   : row["target_mag_N"],
        "target_angle" : row["target_angle_deg"],
        "t_settle"     : t_settle,
        "valid"        : valid,
    })

df_steps = pd.DataFrame(step_records)
n_valid = df_steps["valid"].sum()
print(f"{n_valid}/{len(df_steps)} steps settled within band")
df_steps[["step", "target_mag", "target_angle", "valid", "t_settle"]]

0/28 steps settled within band


,step,target_mag,target_angle,valid,t_settle
0,0,5.0,-128.638,False,None
1,1,10.0,-128.638,False,None
2,2,15.0,-128.638,False,None
3,3,24.0,-128.638,False,None
4,4,5.0,-115.488,False,None
5,5,10.0,-115.488,False,None
6,6,15.0,-115.488,False,None
7,7,24.0,-115.488,False,None
8,8,5.0,-102.338,False,None
9,9,10.0,-102.338,False,None


## 5 — Build combined DataFrame (steady-state windows only)

In [10]:
chunks = []
for _, sr in df_steps[df_steps["valid"]].iterrows():
    mask = (idx_100 >= sr["t_settle"]) & (idx_100 < sr["step_end"])
    ticks = idx_100[mask]
    if len(ticks) == 0:
        continue
    chunk = pd.DataFrame(index=ticks)
    chunk["pos_x"]      = pos_x
    chunk["pos_z"]      = pos_z
    chunk["Fxl"]        = Fxl.reindex(ticks)
    chunk["Fyl"]        = Fyl.reindex(ticks)
    chunk["Fml"]        = Fml.reindex(ticks)
    chunk["Fdl"]        = Fdl.reindex(ticks)
    chunk["Fxg"]        = Fxg.reindex(ticks)
    chunk["Fzg"]        = Fzg.reindex(ticks)
    chunk["Fmg"]        = Fmg.reindex(ticks)
    chunk["Fdg"]        = Fdg.reindex(ticks)
    chunk["mag_error"]  = mag_error.reindex(ticks)
    chunk["dir_error"]  = dir_error.reindex(ticks)
    chunk["me_percent"] = pd.Series(me_percent, index=idx_100).reindex(ticks)
    chunk["de_percent"] = de_percent.reindex(ticks)
    chunk["step"]       = int(sr["step"])
    chunk["target_mag"] = sr["target_mag"]
    chunk["target_angle"] = sr["target_angle"]
    chunks.append(chunk)

df_combined = pd.concat(chunks).rename_axis("timestamp")
print(f"Combined: {len(df_combined)} rows across {len(chunks)} steps")
df_combined.head()

ValueError: No objects to concatenate

## 6 — Save combined CSV

In [ ]:
out_path = os.path.join(OUTPUT_DIR, f"{OUTPUT_NAME}.csv")
df_combined.to_csv(out_path)
print(f"Saved → {out_path}")

## 7 — Quick plots

In [ ]:
# ── Full session overview ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

t_sec = (idx_100 - idx_100[0]).total_seconds()

axes[0].plot(t_sec, Fmg.values, label="Fmg (commanded)", color="#f05050", lw=1)
axes[0].plot(t_sec, Fml.values, label="Fml (measured)",  color="#44cc88", lw=0.8, alpha=0.8)
# mark step starts
for _, sr in df_steps.iterrows():
    t_rel = (sr["step_start"] - idx_100[0]).total_seconds()
    axes[0].axvline(t_rel, color="#ffbb00", lw=0.6, alpha=0.5)
    if sr["valid"]:
        t_s = (sr["t_settle"] - idx_100[0]).total_seconds()
        axes[0].axvline(t_s, color="white", lw=0.6, alpha=0.3, ls="--")
axes[0].set_ylabel("Force (N)"); axes[0].legend(fontsize=8); axes[0].set_title("Magnitude — full session")

axes[1].plot(t_sec, Fdg.values, label="Fdg (commanded)", color="#f05050", lw=1)
axes[1].plot(t_sec, Fdl.values, label="Fdl (measured)",  color="#44cc88", lw=0.8, alpha=0.8)
axes[1].set_ylabel("Direction (°)"); axes[1].set_xlabel("Time (s)")
axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

In [ ]:
# ── Error by target magnitude (steady-state only) ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for mag_val, grp in df_combined.groupby("target_mag"):
    axes[0].scatter([mag_val]*len(grp), grp["mag_error"], s=2, alpha=0.4, label=f"{mag_val}N")
    axes[1].scatter([mag_val]*len(grp), grp["dir_error"], s=2, alpha=0.4, label=f"{mag_val}N")

axes[0].set_xlabel("Target magnitude (N)"); axes[0].set_ylabel("|Fml − Fmg| (N)")
axes[0].set_title("Magnitude error (steady-state)")
axes[1].set_xlabel("Target magnitude (N)"); axes[1].set_ylabel("Direction error (°)")
axes[1].set_title("Direction error (steady-state)")
handles, labels = axes[0].get_legend_handles_labels()
# deduplicate
by_label = dict(zip(labels, handles))
axes[0].legend(by_label.values(), by_label.keys(), markerscale=4, fontsize=8)

plt.tight_layout(); plt.show()

In [ ]:
# ── Per-step summary table ────────────────────────────────────────────────────
summary = df_combined.groupby(["step", "target_mag", "target_angle"]).agg(
    mean_Fml     = ("Fml",        "mean"),
    std_Fml      = ("Fml",        "std"),
    mean_mag_err = ("mag_error",  "mean"),
    mean_dir_err = ("dir_error",  "mean"),
    me_pct       = ("me_percent", "mean"),
    n_samples    = ("Fml",        "count"),
).round(3)
summary